# Hệ Thống Sản Xuất Nội Dung Marketing Tự Động

In [1]:
import os
import json
from typing import Annotated, TypedDict, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

load_dotenv(override=True)

True

In [2]:
# Structure Output
class QuyetDinhSupervisorContent(BaseModel):
    """Quyết định của Supervisor trong hệ thống content."""
    buoc_tiep_theo: str = Field(
        description="Tên bước tiếp theo: 'nghien_cuu_context', 'viet_content', 'HOAN_THANH'"
    )
    ly_do: str = Field(description="Tại sao chọn bước này")
    thong_tin_cho_buoc_tiep: str = Field(
        description="Hướng dẫn và thông tin cụ thể cho bước tiếp theo"
    )

class DanhGiaContent(BaseModel):
    """Kết quả đánh giá nội dung của Evaluator."""
    da_dat_chuan: bool = Field(
        description="True nếu nội dung đã đạt tất cả tiêu chuẩn"
    )
    diem_so: int = Field(description="Điểm từ 1-10", ge=1, le=10)
    diem_manh: str = Field(description="Điểm mạnh của nội dung")
    can_cai_thien: str = Field(
        description="Danh sách cụ thể những gì cần sửa — mỗi điểm một dòng"
    )
    can_them_thong_tin: bool = Field(
        description="True nếu cần người dùng cung cấp thêm thông tin"
    )

In [ ]:
# State
class TrangThaiContentSystem(TypedDict):
    messages: Annotated[list, add_messages]
    # Input từ người dùng
    loai_content: str        # "facebook_post", "email", "instagram_caption"
    thong_tin_san_pham: str  # Thông tin về sản phẩm/dịch vụ
    doi_tuong: str           # Khách hàng mục tiêu
    tieu_chuan: str          # Tiêu chuẩn chất lượng
    # Kết quả từ các bước
    context_da_nghien_cuu: str
    content_hien_tai: str
    # Trạng thái Supervisor
    buoc_hien_tai: str
    # Trạng thái Worker-Evaluator
    phan_hoi_danh_gia: str
    da_hoan_thanh: bool
    so_vong_viet: int
    diem_cao_nhat: int
    # Output cuối
    content_final: str

llm_supervisor = ChatOpenAI(
    model="gpt-4o-mini", temperature=0.1
).with_structured_output(QuyetDinhSupervisorContent)

llm_writer = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

llm_evaluator = ChatOpenAI(
    model="gpt-4o-mini", temperature=0.1
).with_structured_output(DanhGiaContent)

In [4]:
# Node Supervisor
def supervisor_content(state: TrangThaiContentSystem) -> dict:
    """Supervisor điều phối toàn bộ workflow sản xuất content."""
    da_co_context = bool(state.get("context_da_nghien_cuu", ""))
    da_hoan_thanh = state.get("da_hoan_thanh", False)
    so_vong = state.get("so_vong_viet", 0)

    # Logic điều phối đơn giản và rõ ràng
    if da_hoan_thanh or so_vong >= 4:
        return {"buoc_hien_tai": "HOAN_THANH"}

    system = """Bạn là Supervisor của hệ thống sản xuất content marketing.

Quy trình bắt buộc:
1. Đầu tiên luôn phải 'nghien_cuu_context' — hiểu sản phẩm và đối tượng
2. Sau khi có context, chuyển sang 'viet_content' 
3. Chọn 'HOAN_THANH' khi content đã đạt tiêu chuẩn

Hãy phân tích trạng thái và quyết định bước tiếp theo."""

    trang_thai_mo_ta = f"""
Loại content: {state.get('loai_content', '')}
Sản phẩm: {state.get('thong_tin_san_pham', '')}
Đối tượng: {state.get('doi_tuong', '')}
Đã có context nghiên cứu: {'Có' if da_co_context else 'Chưa'}
Số vòng viết: {so_vong}
Content hiện tại: {'Có' if state.get('content_hien_tai') else 'Chưa'}"""

    quyet_dinh = llm_supervisor.invoke([
        SystemMessage(content=system),
        HumanMessage(content=trang_thai_mo_ta)
    ])
    return {
        "buoc_hien_tai": quyet_dinh.buoc_tiep_theo,
        "messages": [AIMessage(
            content=f"[Supervisor] → {quyet_dinh.buoc_tiep_theo}: {quyet_dinh.ly_do}"
        )]
    }

In [5]:
# Node: Nghiên cứu context
def nghien_cuu_context(state: TrangThaiContentSystem) -> dict:
    """Phân tích sản phẩm, đối tượng và định hình tone of voice."""

    system = """Bạn là chuyên gia marketing content với am hiểu sâu về thị trường Việt Nam.
    
Nhiệm vụ: Phân tích và chuẩn bị context để viết content hiệu quả.
Output cần bao gồm:
1. Insight về đối tượng mục tiêu (tâm lý, pain point, mong muốn)
2. Điểm bán hàng nổi bật nhất của sản phẩm (USP)
3. Tone of voice phù hợp (ví dụ cụ thể về từ ngữ nên/không nên dùng)
4. Các từ khóa/hashtag phù hợp nếu dùng cho social media
5. Lưu ý văn hóa/xu hướng đặc thù của người Việt liên quan"""

    ket_qua = llm_writer.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"""
Loại content: {state.get('loai_content')}
Thông tin sản phẩm: {state.get('thong_tin_san_pham')}
Đối tượng mục tiêu: {state.get('doi_tuong')}

Hãy phân tích và chuẩn bị context:""")
    ])

    return {
        "context_da_nghien_cuu": ket_qua.content,
        "message": [AIMessage(content=f"[Context Analysis]\n{ket_qua.content}")]
    }

In [6]:
# Node: Viết content
def viet_content(state: TrangThaiContentSystem) -> dict:
    """Writer tạo nội dung, có tham khảo feedback từ vòng trước nếu có."""
    loai = state.get("loai_content", "post")
    context = state.get("context_da_nghien_cuu", "")
    san_pham = state.get("thong_tin_san_pham", "")
    phan_hoi = state.get("phan_hoi_danh_gia", "")
    so_vong = state.get("so_vong_viet", 0)
    if phan_hoi and so_vong > 0:
        huong_dan_viet_lai = f"""
 ĐÂY LÀ LẦN VIẾT LẠI (Vòng {so_vong + 1})

Lần trước bị reject vì những lý do sau — bạn PHẢI khắc phục tất cả:
{phan_hoi}

Đừng chỉ sửa nhỏ — hãy viết lại hoàn toàn nếu cần để đạt tiêu chuẩn."""
    else:
        huong_dan_viet_lai = "Đây là lần viết đầu tiên — hãy tạo ra nội dung tốt nhất."
    
    # Template khác nhau cho từng loại content
    format_huong_dan = {
        "facebook_post": "Bài Facebook: Có hook mở đầu, body ngắn gọn súc tích, CTA cuối. 150-300 từ. Có thể dùng emoji phù hợp.",
        "email": "Email marketing: Subject line hấp dẫn, preview text, body có structure rõ, CTA nút bấm. 200-400 từ.",
        "instagram_caption": "Caption Instagram: Hook ngắn 1-2 câu đầu (hiển thị trước 'See more'), body, hashtags. Tối đa 150 từ + hashtags."
    }.get(loai, "Nội dung marketing ngắn gọn, súc tích, có CTA.")

    system = f"""Bạn là copywriter chuyên nghiệp với kinh nghiệm viết content marketing Việt Nam.
    
Format yêu cầu: {format_huong_dan}

{huong_dan_viet_lai}"""

    ket_qua = llm_writer.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"""
THÔNG TIN SẢN PHẨM:
{san_pham}

CONTEXT VÀ INSIGHT:
{context}

Viết {loai} ngay bây giờ:""")
    ])

    return {
        "content_hien_tai": ket_qua.content,
        "so_vong_viet": so_vong + 1,
        "messages": [AIMessage(
            content=f"[Writer - Vòng {so_vong + 1}]\n{ket_qua.content}"
        )]
    }

In [7]:
# Node: Evaluator - Đánh giá Content
def danh_gia_content(state: TrangThaiContentSystem) -> dict:
    """Evaluator chấm điểm content theo tiêu chuẩn đã đặt ra."""

    content = state.get("content_hien_tai", "")
    tieu_chuan = state.get("tieu_chuan", "")
    loai = state.get("loai_content", "")
    so_vong = state.get("so_vong_viet", 1)
    phan_hoi_truoc = state.get("phan_hoi_danh_gia", "")

    ghi_chu_lich_su = ""
    if phan_hoi_truoc and so_vong > 1:
        ghi_chu_lich_su = f"""
FEEDBACK VÒNG TRƯỚC: {phan_hoi_truoc}
Nếu Writer vẫn mắc lỗi tương tự, hãy ghi rõ và cân nhắc đặt can_them_thong_tin=True."""

    system = f"""Bạn là Editor cao cấp — người đánh giá content marketing một cách nghiêm khắc và khách quan.

Loại content đang đánh giá: {loai}

TIÊU CHUẨN BẮT BUỘC PHẢI ĐẠT:
{tieu_chuan}

Nguyên tắc đánh giá:
- Chỉ đặt da_dat_chuan=True khi ĐẠT TẤT CẢ tiêu chuẩn
- Feedback phải CỤ THỂ: không viết "viết chưa hay" mà phải viết "câu mở đầu quá dài, cần rút xuống còn 10 từ"
- Điểm số phản ánh thực chất: 7+ chỉ khi thực sự tốt
{ghi_chu_lich_su}"""

    ket_qua = llm_evaluator.invoke([
        SystemMessage(content=system),
        HumanMessage(content=f"""
CONTENT CẦN ĐÁNH GIÁ:
{content}

Đánh giá chi tiết:""")
    ])
    # Cập nhật điểm cao nhất
    diem_cu = state.get("diem_cao_nhat", 0)
    diem_moi = max(diem_cu, ket_qua.diem_so)

    # Nếu đạt — lưu vào content_final
    content_final = state.get("content_final", "")
    if ket_qua.da_dat_chuan:
        content_final = content

    return {
        "da_hoan_thanh": ket_qua.da_dat_chuan,
        "phan_hoi_danh_gia": ket_qua.can_cai_thien,
        "diem_cao_nhat": diem_moi,
        "content_final": content_final
    }

In [8]:
# routing
def dinh_tuyen_supervisor_content(state: TrangThaiContentSystem) -> str:
    buoc = state.get("buoc_hien_tai", "HOAN_THANH")
    if buoc == "HOAN_THANH":
        return END
    return buoc

def dinh_tuyen_sau_danh_gia(state: TrangThaiContentSystem) -> str:
    if state.get("da_hoan_thanh"):
        return END
    if state.get("so_vong_viet", 0) >=4:
        return END
    return "viet_content"

In [10]:
# Build Graph
graph_builder = StateGraph(TrangThaiContentSystem)

# Nodes
graph_builder.add_node("supervisor", supervisor_content)
graph_builder.add_node("nghien_cuu_context", nghien_cuu_context)
graph_builder.add_node("viet_content", viet_content)
graph_builder.add_node("danh_gia_content", danh_gia_content)

# Edges
graph_builder.add_edge(START, "supervisor")

graph_builder.add_conditional_edges(
    "supervisor",
    dinh_tuyen_supervisor_content,
    {
        "nghien_cuu_context": "nghien_cuu_context",
        "viet_content": "viet_content",
        END: END
    }
)

graph_builder.add_edge("nghien_cuu_context", "supervisor")
graph_builder.add_edge("viet_content", "danh_gia_content")
graph_builder.add_conditional_edges(
    "danh_gia_content",
    dinh_tuyen_sau_danh_gia,
    {"viet_content": "viet_content",
    END: END}
)

# Sqlite 
conn = sqlite3.connect("content_system.db", check_same_thread=False)
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver(conn)
graph_content = graph_builder.compile(checkpointer=checkpointer)

In [11]:
# Test
import uuid

def san_xuat_content(
    loai_content: str,
    thong_tin_san_pham: str,
    doi_tuong: str,
    tieu_chuan: str
) -> str:
    """Interface chính để sản xuất content."""

    thread_id = f"content_{str(uuid.uuid4())[:8]}"
    config = {"configurable": {"thread_id": thread_id}}

    state_ban_dau = {
        "messages":              [HumanMessage(content=f"Tạo {loai_content}")],
        "loai_content":          loai_content,
        "thong_tin_san_pham":    thong_tin_san_pham,
        "doi_tuong":             doi_tuong,
        "tieu_chuan":            tieu_chuan,
        "context_da_nghien_cuu": "",
        "content_hien_tai":      "",
        "buoc_hien_tai":         "",
        "phan_hoi_danh_gia":     "",
        "da_hoan_thanh":         False,
        "so_vong_viet":          0,
        "diem_cao_nhat":         0,
        "content_final":         ""
    }

    ket_qua = graph_content.invoke(state_ban_dau, config=config)

    content_cuoi = ket_qua.get("content_final") or ket_qua.get("content_hien_tai")

    print(f"  Số vòng: {ket_qua['so_vong_viet']}")
    print(f"  Điểm cao nhất: {ket_qua['diem_cao_nhat']}/10")
    
    print(" NỘI DUNG CUỐI CÙNG:")
    print(content_cuoi)

    return content_cuoi

In [12]:
# ─── Test Case 1: Facebook Post ───────────────────────────────────────
san_xuat_content(
    loai_content="facebook_post",

    thong_tin_san_pham="""Sản phẩm: Serum Vitamin C 'Sáng Mịn'
Thương hiệu: Herbio (thuần chay, Việt Nam)
Giá: 299.000đ / 30ml
Công dụng chính: Làm sáng da, mờ thâm nám sau 4 tuần
Thành phần nổi bật: Vitamin C 15%, Niacinamide 5%, chiết xuất nghệ vàng
Điểm khác biệt: Không paraben, không cồn, pH cân bằng, phù hợp da nhạy cảm
Ưu đãi hiện tại: Mua 2 tặng 1 đến hết tháng 4""",

    doi_tuong="""Nữ 22-35 tuổi, sống ở đô thị (HCM, HN, Đà Nẵng)
Quan tâm đến skincare tự nhiên, ngại hóa chất
Hay lo lắng về thâm nám sau mụn, da không đều màu
Active trên Facebook, thích đọc review thật từ người dùng thật
Thu nhập trung bình khá, sẵn sàng trả giá hợp lý cho sp chất lượng""",

    tieu_chuan="""Post phải đạt TẤT CẢ:
1. Hook 2 câu đầu phải gây tò mò hoặc chạm đúng nỗi đau
2. Nêu được lợi ích cụ thể (không chỉ nói 'làm sáng da' chung chung)
3. Có bằng chứng/lý do tin tưởng (thành phần, không paraben, v.v.)
4. CTA rõ ràng với ưu đãi
5. Dùng emoji tự nhiên (không lạm dụng)
6. Giọng văn gần gũi như bạn bè chia sẻ, không như quảng cáo cứng
7. Độ dài 180-280 từ"""
)

  Số vòng: 1
  Điểm cao nhất: 8/10
 NỘI DUNG CUỐI CÙNG:
🌟 **Sáng Mịn Da Cùng Herbio!** 🌟

Bạn có biết rằng làn da sáng mịn không chỉ là ước mơ? Với Serum Vitamin C 'Sáng Mịn' từ Herbio, bạn sẽ thấy sự khác biệt chỉ sau 4 tuần! 💖

Chứa nồng độ Vitamin C 15% và Niacinamide 5%, sản phẩm của chúng tôi giúp làm sáng da, mờ thâm nám hiệu quả mà vẫn an toàn cho làn da nhạy cảm. Đặc biệt, không chứa paraben hay cồn, serum này hoàn toàn thuần chay và phù hợp với những ai yêu thích chăm sóc da tự nhiên. 🌿

🌼 **Ưu đãi hấp dẫn**: Mua 2 tặng 1 đến hết tháng 4! Đây là cơ hội tuyệt vời để bạn chăm sóc bản thân mà không lo về giá. 

Hãy để Herbio đồng hành cùng bạn trên hành trình tìm lại làn da rạng rỡ nhé! Đừng chần chừ, click ngay để đặt hàng! 

👉 **#Herbio #SerumSangMin #VitaminC #LamSangDa #MoThamNam #ThuanChay**


"🌟 **Sáng Mịn Da Cùng Herbio!** 🌟\n\nBạn có biết rằng làn da sáng mịn không chỉ là ước mơ? Với Serum Vitamin C 'Sáng Mịn' từ Herbio, bạn sẽ thấy sự khác biệt chỉ sau 4 tuần! 💖\n\nChứa nồng độ Vitamin C 15% và Niacinamide 5%, sản phẩm của chúng tôi giúp làm sáng da, mờ thâm nám hiệu quả mà vẫn an toàn cho làn da nhạy cảm. Đặc biệt, không chứa paraben hay cồn, serum này hoàn toàn thuần chay và phù hợp với những ai yêu thích chăm sóc da tự nhiên. 🌿\n\n🌼 **Ưu đãi hấp dẫn**: Mua 2 tặng 1 đến hết tháng 4! Đây là cơ hội tuyệt vời để bạn chăm sóc bản thân mà không lo về giá. \n\nHãy để Herbio đồng hành cùng bạn trên hành trình tìm lại làn da rạng rỡ nhé! Đừng chần chừ, click ngay để đặt hàng! \n\n👉 **#Herbio #SerumSangMin #VitaminC #LamSangDa #MoThamNam #ThuanChay**"